# INDEX-BASED RAG

We will build a RAG that explains how a DRONE for image analysis, that uses computer vision techniques, works.

In [1]:
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, Document
from llama_index.vector_stores.deeplake import DeepLakeVectorStore
import os
from pathlib import Path
import requests
from bs4 import BeautifulSoup
import re

d:\PersonalStudy\projects\RAG_basics\.venv\Lib\site-packages\deeplake\util\check_latest_version.py:32: UserWarning: A newer version of deeplake (4.5.3) is available. It's recommended that you update to the latest version using `pip install -U deeplake`.
  warnings.warn(


In [2]:
with open('key.txt', 'r') as f:
    GROQ_API_KEY = f.readline().strip()

os.environ['GROQ_API_KEY'] = GROQ_API_KEY

# 1) Collecting & Preparig the documents

In [3]:
data_dir = Path(os.path.join(os.getcwd(), 'data'))
data_dir.mkdir(exist_ok=True)
print(data_dir)

d:\PersonalStudy\projects\RAG_basics\3) CH3_GEENRATIVE_AI\data


In [8]:
#We colelct USEFUL links, but also NOISY ones, to test our RAG.
urls = [
    "https://github.com/VisDrone/VisDrone-Dataset",
    "https://paperswithcode.com/dataset/visdrone",
    "https://openaccess.thecvf.com/content_ECCVW_2018/papers/11133/Zhu_VisDrone-DET2018_The_Vision_Meets_Drone_Object_Detection_in_Image_Challenge_ECCVW_2018_paper.pdf",
    "https://github.com/VisDrone/VisDrone2018-MOT-toolkit",
    "https://en.wikipedia.org/wiki/Object_detection",
    "https://en.wikipedia.org/wiki/Computer_vision",
    "https://en.wikipedia.org/wiki/Convolutional_neural_network",
    "https://en.wikipedia.org/wiki/Unmanned_aerial_vehicle",
    "https://www.faa.gov/uas/",
    "https://www.tensorflow.org/", #NOISE
    "https://pytorch.org/", #NOISE
    "https://keras.io/", #NOISE ...
    "https://arxiv.org/abs/1804.06985",
    "https://arxiv.org/abs/2202.11983",
    "https://motchallenge.net/",
    "http://www.cvlibs.net/datasets/kitti/",
    "https://www.dronedeploy.com/",
    "https://www.dji.com/",
    "https://arxiv.org/",
    "https://openaccess.thecvf.com/",
    "https://roboflow.com/",
    "https://www.kaggle.com/",
    "https://paperswithcode.com/",
    "https://github.com/"
]

In [29]:
#Same preprocessing pipeline of previous chapter
HEADERS = {'User-Agent': 'Mozilla/5.0 (compatible; RAGBasicsBot/2.0)'}

def clean_text(text):
    #Remove references like [1], [2], etc..
    content = re.sub(r'\[\d+\]', '', text)
    content = re.sub(r'[^\w\s\.]', '', content)  # Remove punctuation (except periods)
    return content

def fetch_and_clean(url):
    try:
        response = requests.get(url, headers=HEADERS, timeout=10)
        response.raise_for_status()
        
        # Skip PDFs and non-HTML content
        if 'application/pdf' in response.headers.get('content-type', ''):
            print(f"Skipping PDF: {url}")
            return ''
        
        soup = BeautifulSoup(response.content, 'html.parser')
        
        # Remove script and style tags first
        for tag in soup(['script', 'style']):
            tag.decompose()
        
        main_content = None
        
        # Try Wikipedia first
        main_content = soup.find('div', {'class': 'mw-parser-output'})
        
        # Try arXiv
        if not main_content:
            main_content = soup.find('div', {'class': 'content'})
        
        # Try generic content divs
        if not main_content:
            main_content = soup.find('div', {'class': 'article-content'}) or \
                          soup.find('div', {'class': 'post-content'}) or \
                          soup.find('main') or \
                          soup.find('article')
        
        # Fallback: use body if nothing else found
        if not main_content:
            main_content = soup.find('body')
        
        if not main_content:
            print(f"Warning: could not find main content for {url}, skipping.")
            return ''
        
        # Remove common noise sections
        for section_title in ['References', 'Bibliography', 'External links', 'See also', 'Notes', 'Related links']:
            for section in main_content.find_all('span', id=section_title):
                for sib in section.parent.find_next_siblings():
                    sib.decompose()
                section.parent.decompose()
        
        text = main_content.get_text(separator=' ', strip=True)
        text = clean_text(text)
        return text if text else ''
    
    except requests.exceptions.RequestException as e:
        print(f"Error fetching {url}: {e}")
        return ''

In [4]:
#Now we need to store each doc in a specific file, with relevant names
docs_dir = data_dir / 'text_docs'
docs_dir.mkdir(exist_ok=True)
def store_docs(urls):
    for url in urls:
        clean_text = fetch_and_clean(url)
        doc_split = url.split('/')
        doc_name = doc_split[-1] if doc_split[-1] != '' else doc_split[-2]
        filename = os.path.join(docs_dir, doc_name + '.txt')
        with open(filename, 'w', encoding='utf-8') as f:
            f.write(clean_text)
    
    print(f'All docs have been successfully written in {docs_dir} !')

In [31]:
store_docs(urls)

Skipping PDF: https://openaccess.thecvf.com/content_ECCVW_2018/papers/11133/Zhu_VisDrone-DET2018_The_Vision_Meets_Drone_Object_Detection_in_Image_Challenge_ECCVW_2018_paper.pdf
Error fetching https://www.faa.gov/uas/: 403 Client Error: Forbidden for url: https://www.faa.gov/uas/
All docs have been successfully written in d:\PersonalStudy\projects\RAG_basics\3) CH3_GEENRATIVE_AI\data\text_docs !


In [5]:
# Loads the documents searching for supported types (.txt, .docx, .pdf, etc..)
docs = SimpleDirectoryReader(docs_dir).load_data()
docs[0]

Document(id_='de58b7f4-334b-4ce6-a377-2f000cee8062', embedding=None, metadata={'file_path': 'd:\\PersonalStudy\\projects\\RAG_basics\\3) CH3_GEENRATIVE_AI\\data\\text_docs\\1804.06985.txt', 'file_name': '1804.06985.txt', 'file_type': 'text/plain', 'file_size': 3782, 'creation_date': '2026-03-03', 'last_modified_date': '2026-03-03'}, excluded_embed_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], excluded_llm_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], relationships={}, metadata_template='{key}: {value}', metadata_separator='\n', text_resource=MediaResource(embeddings=None, data=None, text='High Energy Physics  Theory arXiv1804.06985 hepth Submitted on 19 Apr 2018 Title A Near Horizon Extreme Binary Black Hole Geometry Authors Jacob Ciafre  Maria J. Rodriguez View a PDF of the paper titled A Near Horizon Extreme Binary Black Hole Geometry by Jacob Ci

# 2) Creating & Populating a Deep Lake vector store

In [6]:
from llama_index.core import StorageContext

In [7]:
# deeplake 3.x + NumPy 2 compatibility patch applied to:
# .venv/Lib/site-packages/deeplake/util/casting.py
# Fix: use type(scalar) + casting='unsafe' in np.can_cast() to match NumPy 1.x NEP-50 semantics

In [8]:
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
embed_model = HuggingFaceEmbedding(model_name="sentence-transformers/all-MiniLM-L6-v2")

d:\PersonalStudy\projects\RAG_basics\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-03-03 12:32:44,160 - INFO - Load pretrained SentenceTransformer: sentence-transformers/all-MiniLM-L6-v2
2026-03-03 12:32:44,510 - INFO - HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-03-03 12:32:44,518 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/modules.json "HTTP/1.1 200 OK"
2026-03-03 12:32:44,639 - INFO - HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
2026-03-03 12:32:44,647 - INFO - HTTP Request

In [9]:
vector_store_path = str(data_dir / "drone_vector_store")
vector_store = DeepLakeVectorStore(dataset_path=vector_store_path, overwrite=True)
storage_context = StorageContext.from_defaults(vector_store=vector_store)

#Create index over all docs
index = VectorStoreIndex.from_documents(documents=docs, storage_context=storage_context, embed_model=embed_model)

Uploading data to deeplake dataset.


100%|██████████| 137/137 [00:00<00:00, 1956.89it/s]

Dataset(path='d:\PersonalStudy\projects\RAG_basics\3) CH3_GEENRATIVE_AI\data\drone_vector_store', tensors=['text', 'metadata', 'embedding', 'id'])

  tensor      htype      shape      dtype  compression
  -------    -------    -------    -------  ------- 
   text       text      (137, 1)     str     None   
 metadata     json      (137, 1)     str     None   
 embedding  embedding  (137, 384)  float32   None   
    id        text      (137, 1)     str     None   


In [10]:
import deeplake

In [12]:
#load in memory
ds = deeplake.load(vector_store_path)

d:\PersonalStudy\projects\RAG_basics\3) CH3_GEENRATIVE_AI\data\drone_vector_store loaded successfully.



In [13]:
import json
import pandas as pd
import numpy as np

# Assuming 'ds' is your loaded Deep Lake dataset

# Create a dictionary to hold the data
data = {}

# Iterate through the tensors in the dataset
for tensor_name in ds.tensors:
    tensor_data = ds[tensor_name].numpy()

    # Check if the tensor is multi-dimensional
    if tensor_data.ndim > 1:
        # Flatten multi-dimensional tensors
        data[tensor_name] = [np.array(e).flatten().tolist() for e in tensor_data]
    else:
        # Convert 1D tensors directly to lists and decode text
        if tensor_name == "text":
            data[tensor_name] = [t.tobytes().decode('utf-8') if t else "" for t in tensor_data]
        else:
            data[tensor_name] = tensor_data.tolist()

# Create a Pandas DataFrame from the dictionary
df = pd.DataFrame(data)

In [14]:
# Function to display a selected record
def display_record(record_number):
    record = df.iloc[record_number]
    display_data = {
        "ID": record.get("id", "N/A"),
        "Metadata": record.get("metadata", "N/A"),
        "Text": record.get("text", "N/A"),
        "Embedding": record.get("embedding", "N/A")
    }

    # Print the ID
    print("ID:")
    print(display_data["ID"])
    print()

    # Print the metadata in a structured format
    print("Metadata:")
    metadata = display_data["Metadata"]
    if isinstance(metadata, list):
        for item in metadata:
            for key, value in item.items():
                print(f"{key}: {value}")
            print()
    else:
        print(metadata)
    print()

    # Print the text
    print("Text:")
    print(display_data["Text"])
    print()

    # Print the embedding
    print("Embedding:")
    print(display_data["Embedding"])
    print()

# Function call to display a record
rec = 0  # Replace with the desired record number
display_record(rec)

ID:
['732db102-50d3-4672-9ca2-f8834af1ba62']

Metadata:
file_path: d:\PersonalStudy\projects\RAG_basics\3) CH3_GEENRATIVE_AI\data\text_docs\1804.06985.txt
file_name: 1804.06985.txt
file_type: text/plain
file_size: 3782
creation_date: 2026-03-03
last_modified_date: 2026-03-03
_node_content: {"id_": "732db102-50d3-4672-9ca2-f8834af1ba62", "embedding": null, "metadata": {"file_path": "d:\\PersonalStudy\\projects\\RAG_basics\\3) CH3_GEENRATIVE_AI\\data\\text_docs\\1804.06985.txt", "file_name": "1804.06985.txt", "file_type": "text/plain", "file_size": 3782, "creation_date": "2026-03-03", "last_modified_date": "2026-03-03"}, "excluded_embed_metadata_keys": ["file_name", "file_type", "file_size", "creation_date", "last_modified_date", "last_accessed_date"], "excluded_llm_metadata_keys": ["file_name", "file_type", "file_size", "creation_date", "last_modified_date", "last_accessed_date"], "relationships": {"1": {"node_id": "de58b7f4-334b-4ce6-a377-2f000cee8062", "node_type": "4", "metadata": {"

In [17]:
# Ensure 'text' column is of type string
df['text'] = df['text'].astype(str)
# Create documents with IDs
documents = [Document(text=row['text'], doc_id=str(row['id'])) for _, row in df.iterrows()]

In [15]:
#ID: useful to organize chunks of text to tranform into nodes
#Text: automatically chunked by deeplake, we can customize this

# 3) Index-Based RAG

We will implement:
- Vector Store Index Engine
- Tree Index
- List Index
- Keyword Table Index

In [ ]:
user_input = "How do drones identify vehicles?"

args = {
    'k':3,
    'temperature':0.1,
    'mt': 1024, #num_output tokens
}


In [18]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('all-MiniLM-L6-v2')


2026-03-03 12:57:28,698 - INFO - Use pytorch device_name: cuda:0
2026-03-03 12:57:28,699 - INFO - Load pretrained SentenceTransformer: all-MiniLM-L6-v2
2026-03-03 12:57:28,866 - INFO - HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-03-03 12:57:28,873 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/modules.json "HTTP/1.1 200 OK"
2026-03-03 12:57:28,998 - INFO - HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
2026-03-03 12:57:29,004 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/config_sentence_transformers.json "HTTP/1.1 200 OK"
2026-03-03 12:57:29,124 - INFO - HTT

In [24]:
def calculate_cosine_similarity_with_text(text1, text2):
    embedding1 = model.encode(text1)
    embedding2 = model.encode(text2)
    cosine_simil = cosine_similarity([embedding1], [embedding2])
    return cosine_simil[0,0]

### VECTOR STORE INDEX QUERY ENGINE

Create embeddings with a defualt embedding model and retrieve top-k with cosine_similarity

In [21]:
vector_store_index = index #already done
print(type(vector_store_index))

<class 'llama_index.core.indices.vector_store.base.VectorStoreIndex'>


In [ ]:
from llama_index.llms.groq import Groq

llm = Groq(
    model="llama-3.3-70b-versatile",   
    api_key=GROQ_API_KEY,
    temperature=args['temperature'],
    max_tokens=args['mt'],
)

In [ ]:
vector_query_engine = vector_store_index.as_query_engine(
    similarity_top_k=args['k'],
    llm=llm,
) #does the same we already did in last chapter, takes a query, search for top_k and gives a response on that context

In [40]:
import pandas as pd
import textwrap
import time
import numpy as np

In [29]:
def index_query(query):
    response = vector_query_engine.query(query)
    print(textwrap.fill(str(response), 100))
    node_data = []
    for node_with_score in response.source_nodes:
        node = node_with_score.node
        node_info = { #top_k chunks
            'Node ID': node.id_,
            'Score': node_with_score.score,
            'Text': node.text
        }
        node_data.append(node_info)
    
    df = pd.DataFrame(node_data)

    return df, response

In [31]:
start_time = time.time()
df, response = index_query(user_input)
end_time = time.time()

elapsed_time = end_time - start_time
print(f'Query execution time: {elapsed_time:4f} seconds')
print(df.to_markdown(index=False, numalign = "left", stralign = "left"))

2026-03-03 13:13:53,713 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


Drones can identify vehicles through various methods, including the use of deep learning-based
machine learning algorithms, which have become increasingly accurate for automatic tracking and
detection of vehicles. Additionally, reidentification methods can be used to automatically identify
vehicles across different cameras with different viewpoints and hardware specifications.
Query execution time: 0.620589 seconds
| Node ID                              | Score    | Text                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                             

How we can see, we have full transparency on which chunks the llm relied to in its response! From the id we can obtain the full chunk

In [32]:
response.source_nodes[0].node_id

'5760c511-8abb-429c-bcb7-7112d190d9ea'

In [33]:
response.source_nodes[0].get_text()

'Automatic tracking and detection of UAVs from commercial cameras have become accurate thanks to the development of deep learning based machine learning algorithms.  248  It is also possible to automatically identify UAVs across different cameras with different viewpoints and hardware specification with reidentification methods.  249  Commercial systems such as the Aaronia AARTOS have been installed on major international airports.  250   251  Once a UAV is detected it can be countered with kinetic force missiles projectiles or another UAV or by nonkinetic force laser microwaves communications jamming.  252  Antiaircraft missile systems such as the Iron Dome are also being enhanced with CUAS technologies. Utilising a smart UAV swarm to counter one or more hostile UAVs is also proposed.  253  A variety of counterUAS CUAS systems have been developed globally to address the growing threat of small and tactical UAVs. These include multilayered approaches combining radar electrooptical sens

In [37]:
for node in response.source_nodes:
    print(node.get_score())

0.5709589719772339
0.5685218572616577
0.51724773645401


In [ ]:
for node in response.source_nodes:
    print(f'CHUNK ID: {node.node_id}, CHUNK SIZE: {len(node.get_text())} chars')
    #we can optimize the chunk size or, like in this case, let llamaindex do it automatically

CHUNK ID: 5760c511-8abb-429c-bcb7-7112d190d9ea, CHUNK SIZE: 4841 chars
CHUNK ID: 17c8adfe-44b2-4239-9add-2ffa386b5a3a, CHUNK SIZE: 3063 chars
CHUNK ID: 097aaff7-a1b0-4745-bd64-f026dac7d019, CHUNK SIZE: 3274 chars


We can also compute some **performance metrics**

In [61]:
def info_metrics(response):
    scores = [node.score for node in response.source_nodes if node.score is not None]
    if scores:
        weights = np.exp(scores) / np.sum(np.exp(scores))
        perf = np.average(scores, weights=weights) / elapsed_time #we want high scores and low execution time
    else:
        perf = 0
    
    average_score = np.average(scores, weights=weights)
    print(f"Average score: {average_score:.4f}")
    print(f"Query execution time: {elapsed_time:.4f} seconds")
    print(f"Performance metric: {perf:.4f} ")

In [62]:
info_metrics(response)

Average score: 0.5529
Query execution time: 0.6206 seconds
Performance metric: 0.8908 


### TREE QUERY ENGINE

In [64]:
from llama_index.core import TreeIndex
tree_index = TreeIndex.from_documents(docs, llm=llm)
print(type(tree_index))

2026-03-03 13:35:06,377 - INFO - > Building index from nodes: 13 chunks
2026-03-03 13:35:08,808 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-03-03 13:35:10,926 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-03-03 13:35:12,933 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-03-03 13:35:15,045 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-03-03 13:35:15,070 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
2026-03-03 13:35:15,071 - INFO - Retrying request to /chat/completions in 14.000000 seconds
2026-03-03 13:35:30,951 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-03-03 13:35:30,980 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 429

<class 'llama_index.core.indices.tree.base.TreeIndex'>


In [66]:
tree_query_engine = tree_index.as_query_engine(similarity_top_k=args['k'], llm=llm,)

In [68]:
# Start the timer
start_time = time.time()
response = tree_query_engine.query(user_input)
# Stop the timer
end_time = time.time()
# Calculate and print the execution time
elapsed_time = end_time - start_time
print(f"Query execution time: {elapsed_time:.4f} seconds")
print(textwrap.fill(str(response), 100))

2026-03-03 13:43:24,672 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-03-03 13:43:24,673 - INFO - >[Level 0] Selected node: [2]/[2]
2026-03-03 13:43:25,623 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-03-03 13:43:25,625 - INFO - >[Level 1] Selected node: [2]/[2]
2026-03-03 13:43:26,434 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-03-03 13:43:26,435 - INFO - >[Level 2] Selected node: [9]/[9]
2026-03-03 13:43:27,033 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


Query execution time: 2.9750 seconds
Drones identify vehicles through a process involving computer vision and machine learning
algorithms, which are applied to the visual data collected by the drone's camera. The VisDrone
dataset provides a large-scale benchmark with carefully annotated ground truth for various computer
vision tasks, including object detection in images and videos. This dataset contains more than 2.6
million bounding boxes of targets of frequent interests, such as pedestrians, cars, bicycles, and
tricycles, which can be used to train and test object detection models. These models can then be
used to detect vehicles in images and videos captured by drones.


In [69]:
similarity_score = calculate_cosine_similarity_with_text(user_input, str(response))
print(f"Cosine Similarity Score: {similarity_score:.3f}")
print(f"Query execution time: {elapsed_time:.4f} seconds")
performance=similarity_score/elapsed_time
print(f"Performance metric: {performance:.4f}")

Batches: 100%|██████████| 1/1 [00:00<00:00, 76.93it/s]

Cosine Similarity Score: 0.710
Query execution time: 2.9750 seconds
Performance metric: 0.2388


### List index query enigne

In [71]:
from llama_index.core import ListIndex
list_index = ListIndex.from_documents(docs)
print(type(list_index))

<class 'llama_index.core.indices.list.base.SummaryIndex'>


In [91]:
with open('key.txt', 'r') as f:
    GROQ_API_KEY = f.readline().strip()

os.environ['GROQ_API_KEY'] = GROQ_API_KEY

llm = Groq(
    model="meta-llama/llama-4-scout-17b-16e-instruct",   
    api_key=GROQ_API_KEY,
    temperature=args['temperature'],
    max_tokens=args['mt'],
)

In [84]:
list_query_engine = list_index.as_query_engine(similarity_top_k = args['k'], llm=llm)

In [85]:
#start the timer
start_time = time.time()
response = list_query_engine.query(user_input)
# Stop the timer
end_time = time.time()
# Calculate and print the execution time
elapsed_time = end_time - start_time
print(f"Query execution time: {elapsed_time:.4f} seconds")

print(textwrap.fill(str(response), 100))

2026-03-03 14:14:09,343 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-03-03 14:14:09,501 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-03-03 14:14:09,558 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
2026-03-03 14:14:09,558 - INFO - Retrying request to /chat/completions in 5.000000 seconds
2026-03-03 14:14:15,087 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-03-03 14:14:15,112 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
2026-03-03 14:14:15,113 - INFO - Retrying request to /chat/completions in 6.000000 seconds
2026-03-03 14:14:21,485 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-03-03 14:14:21,539 - INFO - HTTP Request: POST https://api.groq.com/openai/

Query execution time: 1956.5363 seconds
**Repeat**


### KEYWORD INDEX QUERY ENGINE

In [92]:
from llama_index.core import KeywordTableIndex
keyword_index = KeywordTableIndex.from_documents(docs, llm=llm)
print(type(keyword_index))

2026-03-03 15:34:09,629 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-03-03 15:34:09,848 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-03-03 15:34:09,948 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-03-03 15:34:10,100 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-03-03 15:34:10,373 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-03-03 15:34:10,750 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-03-03 15:34:11,108 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-03-03 15:34:11,291 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-03-03 15:34:11,834 - INFO - HTTP Request: POST http

<class 'llama_index.core.indices.keyword_table.base.KeywordTableIndex'>


In [94]:
data = []
for keyword, doc_ids in keyword_index.index_struct.table.items():
    data.append({'key':keyword, 'doc_ids':doc_ids})
data_df = pd.DataFrame(data)
data_df


,key,doc_ids
0,horizon,{28152a2a-e5e5-4a27-aed6-8e7c737f6ad6}
1,holes,{28152a2a-e5e5-4a27-aed6-8e7c737f6ad6}
2,i will focus on terms that are likely to be si...,"{28152a2a-e5e5-4a27-aed6-8e7c737f6ad6, ede9876..."
3,extract,"{ad70603c-6536-4bc8-9e54-d9a9fc626b16, 135152d..."
4,physical,{28152a2a-e5e5-4a27-aed6-8e7c737f6ad6}
...,...,...
1951,still,{f0a58928-593b-4a55-bfca-05ca29793ada}
1952,pdf,{f0a58928-593b-4a55-bfca-05ca29793ada}
1953,indicated,{f0a58928-593b-4a55-bfca-05ca29793ada}
1954,generative**,{f0a58928-593b-4a55-bfca-05ca29793ada}


In [95]:
keyword_query_engine = keyword_index.as_query_engine(similarity_top_k = args['k'], llm=llm)

In [96]:
import time

# Start the timer
start_time = time.time()

# Execute the query (using .query() method)
response = keyword_query_engine.query(user_input)

# Stop the timer
end_time = time.time()

# Calculate and print the execution time
elapsed_time = end_time - start_time
print(f"Query execution time: {elapsed_time:.4f} seconds")

print(textwrap.fill(str(response), 100))

2026-03-03 15:52:28,678 - INFO - > Starting query: How do drones identify vehicles?
2026-03-03 15:52:28,908 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-03-03 15:52:28,910 - INFO - query keywords: ['machine', 'know', 'anything', 'aerial', 'object detection', 'else', 'surveillance', 'computer', 'recognition', 'sensor systems', 'vision', 'sensor', 'identification', 'let', 'learning', 'image', 'tracking', 'tracking \n\nlet me know if i can help with anything else!', 'systems', 'machine learning', 'image recognition', 'drone technology', 'help', 'detection', 'vehicle identification', 'computer vision', 'technology', 'object', 'aerial surveillance', 'vehicle', 'drones', 'drone']
2026-03-03 15:52:28,911 - INFO - > Extracted keywords: ['machine', 'know', 'anything', 'aerial', 'object detection', 'else', 'surveillance', 'computer', 'recognition', 'vision', 'sensor', 'identification', 'let', 'learning', 'image', 'tracking', 'systems', 'machi

Query execution time: 2.8054 seconds
Drones identify vehicles through image recognition, which involves classifying detected objects into
categories. This process includes estimating application-specific parameters, such as object pose or
size, and verifying that the object satisfies model-based and application-specific assumptions.
Image understanding systems, operating on multiple levels of abstraction, facilitate this
classification.


In [99]:
similarity_score = calculate_cosine_similarity_with_text(user_input, str(response))
print(f"Cosine Similarity Score: {similarity_score:.3f}")
print(f"Query execution time: {elapsed_time:.4f} seconds")
performance=similarity_score/elapsed_time
print(f"Performance metric: {performance:.4f}")

Batches: 100%|██████████| 1/1 [00:00<00:00, 41.66it/s]

Cosine Similarity Score: 0.752
Query execution time: 2.8054 seconds
Performance metric: 0.2681


In [108]:

# Retrieve keywords that matched the response source nodes
node_ids = {node.node.node_id for node in response.source_nodes}
matched_keywords = [kw for kw, ids in keyword_index.index_struct.table.items() if ids & node_ids]
print(f"Matched keywords ({len(matched_keywords)}): {matched_keywords}")


Matched keywords (285): ['high', 'physics', 'provided', 'text', 'related', 'terms', 'document', 'keywords', 'stopwords', 'giaotracker', 'know', 'extracted', 'quality', 'anything', 'help', 'requested', 'models', 'videos', 'global', 'else', 'computer vision', 'deep', 'here are the extracted keywords in the requested format:\n\nkeywords: giaotracker', 'video quality', 'video', 'object', 'computer', 'deep models', "global information\n\nlet me know if you'd like me to help with anything else!", 'vision', 'let', 'optimizing strategies', 'drone videos', 'tracking', 'visdrone', 'object tracking', 'information', 'strategies', 'optimizing', 'drone', 'format', 'like', 'mcmot', 'machine', 'concepts', 'note', 'scene', 'scene reconstruction', "etc. and focused on extracting relevant technical terms and concepts from the text. let me know if you'd like me to adjust or expand on this list!", 'recognition', 'list', 'technical', 'learning', 'image', 'intelligent', 'reconstruction', 'expand', 'extractin